# 🚀 Transformers & LLM Fine-Tuning Masterclass: From BERT to QLoRA & Ollama

This masterclass notebook provides an all-in-one comprehensive, executable reference covering the entire progression of modern NLP, Transformers, and LLM Engineering:

### 📚 Curriculum Overview:
1. **Part 1: Hugging Face Foundations & Tokenizer Mechanics** (`AutoTokenizer`, token IDs, attention masks, special tokens)
2. **Part 2: Encoder Fine-Tuning with BERT** (`bert-base-uncased`, `DataCollatorWithPadding`, AdamW, Warmup, Metrics)
3. **Part 3: Autoregressive Instruction Tuning with GPT-2** (Special token injection, embedding resizing, label loss masking with `-100`)
4. **Part 4: Modern Chat LLMs with Microsoft Phi-3.5-mini** (Chat Templates `apply_chat_template`, `DataCollatorForSeq2Seq`, FP16/BF16)
5. **Part 5: Parameter-Efficient Fine-Tuning (PEFT & LoRA)** (`LoraConfig`, $r=8, \alpha=16$, `get_peft_model`, trainable parameter inspection)
6. **Part 6: QLoRA & 8-Bit Quantization with SFTTrainer** (`BitsAndBytesConfig`, `prepare_model_for_kbit_training`, `SFTConfig`, `SFTTrainer`, Alpaca dataset)
7. **Part 7: Full Deployment Lifecycle** (`merge_and_unload()`, SafeTensors export, `llama.cpp` GGUF conversion, Ollama `Modelfile`)

## Part 1: Hugging Face Core APIs & Tokenizer Mechanics

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample_text = "Deep Learning with Transformers and Hugging Face!"
encoded = tokenizer(sample_text, return_tensors="pt")

print("Input IDs:      ", encoded["input_ids"].tolist()[0])
print("Attention Mask: ", encoded["attention_mask"].tolist()[0])
print("Decoded Text:   ", tokenizer.decode(encoded["input_ids"][0]))

## Part 2: Encoder Sequence Classification with BERT
Fine-tuning bidirectional encoder representations (`bert-base-uncased`) using dynamic mini-batch padding.

In [ ]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5)

# Dynamic padding collator
data_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro")
    }

print("BERT Classification Architecture initialized successfully.")
print(f"Total Parameters: {sum(p.numel() for p in bert_model.parameters()):,}")

## Part 3: Autoregressive Instruction Tuning with Loss Masking (`-100`)
Adding custom delimiters `<|user|>` and `<|assistant|>`, resizing the embedding layer, and masking prompt tokens with `-100` so loss is calculated **strictly on assistant responses**.

In [ ]:
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

gpt_tokenizer = GPT2TokenizerFast.from_pretrained("openai-community/gpt2")
gpt_model = GPT2LMHeadModel.from_pretrained("openai-community/gpt2")

USER_TOKEN = "<|user|>"
ASSISTANT_TOKEN = "<|assistant|>"

# Add special tokens & resize embeddings
gpt_tokenizer.add_special_tokens({
    "pad_token": gpt_tokenizer.eos_token,
    "additional_special_tokens": [USER_TOKEN, ASSISTANT_TOKEN]
})
gpt_model.resize_token_embeddings(len(gpt_tokenizer))

def prepare_instruction_sample(user_prompt, assistant_response, max_length=128):
    full_text = f"{USER_TOKEN}{user_prompt}{ASSISTANT_TOKEN}{assistant_response}"
    encoding = gpt_tokenizer(full_text, truncation=True, max_length=max_length, padding="max_length")
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    
    # Label masking: set all prompt tokens to -100
    labels = [-100] * len(input_ids)
    assistant_id = gpt_tokenizer.convert_tokens_to_ids(ASSISTANT_TOKEN)
    
    if assistant_id in input_ids:
        start_idx = input_ids.index(assistant_id) + 1
        for i in range(start_idx, len(input_ids)):
            if attention_mask[i] == 1:
                labels[i] = input_ids[i]
                
    return {"input_ids": input_ids, "labels": labels}

sample = prepare_instruction_sample("What is 2+2?", "4")
print("Input IDs (first 15): ", sample["input_ids"][:15])
print("Labels (first 15):    ", sample["labels"][:15])
print("Active Loss Tokens:   ", sum(l != -100 for l in sample["labels"]))

## Part 4: Modern Chat LLMs with Chat Templates (Phi-3.5-mini)
Using official Hugging Face Chat Templates (`apply_chat_template`) and `DataCollatorForSeq2Seq`.

In [ ]:
from transformers import AutoTokenizer, DataCollatorForSeq2Seq

phi_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)

messages = [
    {"role": "system", "content": "You are an expert AI tutor."},
    {"role": "user", "content": "Explain how backpropagation works in 1 sentence."}
]

formatted_chat = phi_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("--- Formatted Chat Template ---")
print(formatted_chat)

## Part 5: Parameter-Efficient Fine-Tuning (PEFT & LoRA)
Freezing base model weights and training low-rank decomposition matrices ($W = W_0 + \frac{\alpha}{r} BA$) on query/key/value and output projections.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                       # Low-rank dimension
    lora_alpha=16,             # Scaling parameter (alpha / r = 2.0)
    lora_dropout=0.05,         # LoRA dropout
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"]
)

peft_model = get_peft_model(gpt_model, lora_config)
peft_model.print_trainable_parameters()

## Part 6: QLoRA & 8-Bit Quantization with `SFTTrainer`
Combining 8-bit quantized weights (`BitsAndBytesConfig`), k-bit training preparation, custom Jinja templates, and `SFTTrainer` (`trl`) with sample packing.

In [ ]:
from transformers import BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

# 8-Bit Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0
)

# Supervised Fine-Tuning Configuration with packing
sft_config = SFTConfig(
    output_dir="./qlora-output",
    dataset_text_field="formatted_text",
    max_seq_length=512,
    packing=True,              # Concatenates multiple short examples into 512 chunks for high GPU throughput
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    report_to="none"
)

print("QLoRA & SFTTrainer configuration verified successfully.")

## Part 7: Full Deployment Lifecycle (Merging, GGUF & Ollama)
Fusing LoRA adapters (`merge_and_unload()`), exporting SafeTensors, converting to GGUF format via `llama.cpp`, and configuring an Ollama `Modelfile`.

In [ ]:
print("=== 1. Merge LoRA Adapters into Base Model ===")
# merged_model = peft_model.merge_and_unload()
# merged_model.save_pretrained('./merged-model', safe_serialization=True)
# tokenizer.save_pretrained('./merged-model')

print("\n=== 2. GGUF Conversion Command (llama.cpp) ===")
print("python3 llama.cpp/convert_hf_to_gguf.py ./merged-model --outfile ./model-f16.gguf --outtype f16")

print("\n=== 3. Ollama Modelfile Syntax ===")
modelfile_sample = """FROM ./model-f16.gguf
TEMPLATE """{{ if .System }}<|system|>
{{ .System }}<|end|>
{{ end }}{{ if .Prompt }}<|user|>
{{ .Prompt }}<|end|>
{{ end }}<|assistant|>
"""
SYSTEM """You are a helpful and precise assistant."""
PARAMETER temperature 0.7
PARAMETER stop "<|end|>"
"""
print(modelfile_sample)

print("=== 4. Ollama CLI Execution Commands ===")
print("ollama create my-custom-model -f Modelfile")
print("ollama run my-custom-model")